# Thesis Notebook — SRQ2/SRQ3 (Agentic A/B/C Evaluation)

**Owner**: Enrico Manfron
**Institution**: Copenhagen Business School (MSc 2026)
**Created**: 2026-04-25

**Research questions answered**:
- **SRQ2**: Does adding ML predictive models to an AI agent improve forecast accuracy and recommendation quality vs an LLM-only baseline?
- **SRQ3**: How does our open-source AI+ML approach compare to a commercial industrial system (Prometheus by Manifold)?

**Design**: A/B/C between three systems on **50 stratified prompts**:
- **System A** — `LLM-only` (GPT-4o-mini, no tools, baseline)
- **System B** — `LLM + ML` (GPT-4o-mini + 5 specialized models via OpenAI function calling)
- **System C** — `Prometheus` (Manifold's industrial AI, integrated via API once available)

**Evaluation framework** (12 metrics + pairwise + composite + radar):

| Layer | Metrics |
|-------|---------|
| Quantitative auto | MAPE, MAE, RMSE, sMAPE, Hallucination rate |
| Operational auto | Latency p50/p95, Cost per query |
| Qualitative judge | Groundedness, Actionability, Specificity, Coherence, Business relevance |
| Comparative | Pairwise preference (win rate) |

Plus a **mini human eval** on 10 prompts (validates LLM judge).

**Costs**: ~$6 total (OpenAI ~$0.10 + Anthropic Sonnet judge ~$5).

**Output**: `outputs_ab_test/` — predictions, judgments, figures for Chapter 7.

---

# §0 — Setup

**Why**: imports, OpenAI + Anthropic clients, paths, load the 5 specialized ML models
(reused from the SRQ1 phase). Run before everything else.

In [1]:
# §0 — Setup
import os, sys, json, time, pickle, warnings
from pathlib import Path
from typing import Optional, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path("/Users/enricomanfron/Desktop/Thesis Maniflod")
sys.path.insert(0, str(PROJECT_ROOT))
ANALYSIS_DIR = PROJECT_ROOT / "thesis" / "analysis"
OUTPUT_DIR = ANALYSIS_DIR / "outputs" / "ab_test_v5"
FIGURE_DIR = OUTPUT_DIR / "figures"
for d in (OUTPUT_DIR, FIGURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Load .env
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip("\"").strip("\'")

# OpenAI + Anthropic SDKs
from openai import OpenAI
import anthropic

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

OPENAI_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "claude-sonnet-4-6"

# Pricing (USD per 1M tokens, Apr 2026)
PRICING = {
    "gpt-4o-mini": {"input": 0.150, "output": 0.600},
    "claude-sonnet-4-6": {"input": 3.0, "output": 15.0},
}

SEED = 42
np.random.seed(SEED)

# ── Load 5 specialized ML models from Phase 2 ──────────────────────────────
CATEGORY_MODELS = {
    "csd":          {"out": ANALYSIS_DIR / "outputs",              "primary": "xgboost",  "fm": "feature_matrix_split.parquet",   "mape": 16.77},
    "danskvand":    {"out": ANALYSIS_DIR / "outputs_danskvand",    "primary": "lightgbm", "fm": "feature_matrix_split.parquet",   "mape": 21.23},
    "energidrikke": {"out": ANALYSIS_DIR / "outputs_energidrikke", "primary": "xgboost",  "fm": "feature_matrix_split.parquet",   "mape": 21.18},
    "rtd":          {"out": ANALYSIS_DIR / "outputs_rtd_v2",       "primary": "lightgbm", "fm": "feature_matrix_split_v2.parquet","mape": 25.56},
    "totalbeer":    {"out": ANALYSIS_DIR / "outputs_totalbeer",    "primary": "xgboost",  "fm": "feature_matrix_split.parquet",   "mape": 26.65},
}

_model_cache = {}
def load_model_for_category(cat):
    if cat in _model_cache: return _model_cache[cat]
    cfg = CATEGORY_MODELS[cat]
    with open(cfg["out"] / "pipelines" / f"model_{cfg['primary']}.pkl", "rb") as f:
        model = pickle.load(f)
    with open(cfg["out"] / "pipelines" / "pipe_tree.pkl", "rb") as f:
        pipe = pickle.load(f)
    fm = pd.read_parquet(cfg["out"] / cfg["fm"])
    art = {"model": model, "pipe": pipe, "fm": fm, "model_type": cfg["primary"], "mape": cfg["mape"]}
    _model_cache[cat] = art
    return art

# Verify everything loads
print("Setup OK:")
print(f"  OpenAI model    : {OPENAI_MODEL}")
print(f"  Judge model     : {JUDGE_MODEL}")
for cat in CATEGORY_MODELS:
    art = load_model_for_category(cat)
    n_brands = art["fm"]["brand"].nunique()
    n_test = (art["fm"]["split"] == "test").sum()
    print(f"  {cat:<14s} {art['model_type']:<10s} {n_brands} brands, {n_test:,} test rows, expected MAPE {art['mape']}%")


Setup OK:
  OpenAI model    : gpt-4o-mini
  Judge model     : claude-sonnet-4-6
  csd            xgboost    77 brands, 3,940 test rows, expected MAPE 16.77%
  danskvand      lightgbm   25 brands, 1,175 test rows, expected MAPE 21.23%
  energidrikke   xgboost    26 brands, 1,580 test rows, expected MAPE 21.18%
  rtd            lightgbm   42 brands, 1,691 test rows, expected MAPE 25.56%
  totalbeer      xgboost    246 brands, 8,995 test rows, expected MAPE 26.65%


In [16]:
# Recovery save with bullet-proof conversion
import pandas as pd

df_resp = pd.DataFrame(all_responses)
for col in df_resp.select_dtypes(include=["object"]).columns:
    df_resp[col] = df_resp[col].apply(lambda v: str(v) if isinstance(v, (dict, list)) else v)
    df_resp[col] = df_resp[col].astype(str).replace("None", "")
df_resp["predicted_sales_units"] = pd.to_numeric(df_resp["predicted_sales_units"], errors="coerce")

results_path = OUTPUT_DIR / "raw_responses.parquet"
df_resp.to_parquet(results_path, index=False)
print(f"✅ Saved {len(df_resp)} responses to {results_path.name}")
print(f"\nResponses per (system, archetype):")
print(df_resp.groupby(["system", "archetype"]).size().unstack(fill_value=0))


✅ Saved 98 responses to raw_responses.parquet

Responses per (system, archetype):
archetype  anomaly  driver  forecast  recommendation
system                                              
A               10      14        15              10
B               10      14        15              10


### §0 — Observations + Decisions

- All 5 models loaded? _..._
- OpenAI + Anthropic API keys working? _..._

---

# §1 — Definition of 50 stratified prompts

**Why**: 4 prompt archetypes × 5 categories × balanced mix = 50 total. Each prompt
includes ground-truth metadata (target row in test set) for forecast-accuracy
evaluation.

**Archetypes**:
1. **Forecast** (3/cat): "Predict X at Y in Z" — measures numeric accuracy
2. **Driver** (3/cat): "What will drive X at Y in Z?" — measures groundedness/coherence
3. **Recommendation** (2/cat): "What should we do about X at Y in Z?" — measures actionability
4. **Anomaly** (2/cat): "X dropped K% at Y. What happened?" — measures diagnostic reasoning

In [ ]:
# §1 v4 (Phase 5) — Load 50 forecast-only prompts from disk
# Prompts are generated externally by /tmp/generate_phase5_prompts.py
# 11 archetypes, 10 per category × 5 categories, real test-set tuples (Sep-Nov 2025)

prompts_df = pd.read_csv(OUTPUT_DIR / "prompts.csv")
print(f"Loaded {len(prompts_df)} prompts from {OUTPUT_DIR / 'prompts.csv'}")
print(f"\nDistribution by archetype:")
print(prompts_df["archetype"].value_counts().to_string())
print(f"\nDistribution by category:")
print(prompts_df["category"].value_counts().to_string())
print(f"\nFirst 3 prompts:")
for _, r in prompts_df.head(3).iterrows():
    print(f"  [{r['id']:>2d}] {r['archetype']:<22s} {r['category']:<13s} | actual={r['actual_sales_units']:,.0f}")
    print(f"        {r['query'][:140]}")


### §1 — Observations + Decisions

- _Total prompts: ..._
- _Prompts per category: balanced?_
- _Any category with insufficient test data? Adjust if needed._

---

# §2 — System A: LLM-only (baseline, no tools)

**Why**: the baseline. GPT-4o-mini with no access to our ML models. Answers from
pre-training knowledge alone. This is the comparison floor — anything System B/C
adds must beat this.

In [ ]:
# §2 v3 — System A (LLM-only, gpt-4o, with few-shot quantified output)

OPENAI_MODEL = "gpt-4o"
PRICING["gpt-4o"] = {"input": 2.50, "output": 10.0}

SYSTEM_A_PROMPT_V3 = """You are Prometheus, an AI Virtual Colleague at Royal Unibrew specializing in beverage strategic advice for the Danish market.

You have NO TOOLS. You cannot access historical data or any ML model. You can only use general industry knowledge.

CRITICAL HONESTY RULES:
- For specific sales forecasts: set predicted_sales_units to null. NEVER invent sales numbers.
- For directional advice: you can suggest WHAT data to retrieve, WHO to consult, WHAT to investigate

RESPONSE SCHEMA (strict JSON):
{
  "predicted_sales_units": null,
  "reasoning": "<honest explanation that you cannot forecast without data/model access; suggest what should be done to answer the query (consult Nielsen DB, sales team, trade marketing calendar, etc.)>",
  "recommendations": "<directional advice grounded in industry knowledge: which data to retrieve, which stakeholders to consult, what to investigate>"
}

ALLOWED in recommendations:
- "Recommend retrieving last 12-24 months of actuals for [brand] at [channel] from Nielsen DB"
- "Consult sales team field intel for recent trade activity at this retailer"
- "Triangulate with: (1) Nielsen DB, (2) trade marketing calendar, (3) competitor scan"
- General directional reasoning ("if this is a mature brand, expect single-digit growth")

NOT allowed:
- Specific sales unit forecasts
- DKK budgets / euro amounts
- ROI multipliers
- % uplift estimates from marketing actions"""

def system_A(query: str) -> dict:
    t0 = time.perf_counter()
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_A_PROMPT_V3},
            {"role": "user", "content": query},
        ],
        response_format={"type": "json_object"},
        max_tokens=6000,
    )
    latency = time.perf_counter() - t0
    usage = response.usage
    cost = (usage.prompt_tokens * PRICING[OPENAI_MODEL]["input"] +
            usage.completion_tokens * PRICING[OPENAI_MODEL]["output"]) / 1_000_000
    try:
        parsed = json.loads(response.choices[0].message.content)
    except Exception:
        parsed = {"predicted_sales_units": None, "reasoning": response.choices[0].message.content, "recommendations": ""}
    return {
        "system": "A",
        "raw_text": response.choices[0].message.content,
        "predicted_sales_units": parsed.get("predicted_sales_units"),
        "reasoning": parsed.get("reasoning", ""),
        "recommendations": parsed.get("recommendations", ""),
        "latency_s": latency,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "cost_usd": cost,
    }

print("L0 system (LLM-only baseline) defined with 10k tokens + actionability layer.")
print("Smoke test (Phase 5 forecast prompt):")
test = system_A("Predict the monthly sales (units) for CARLSBERG at MENY in October 2025.")
print(f"  predicted: {test['predicted_sales_units']}")
print(f"  reasoning[:300]: {test.get('reasoning','')[:300]}...")
print(f"  recommendations[:300]: {test['recommendations'][:300]}...")
print(f"  cost: ${test['cost_usd']:.4f}")


### §2 — Observations + Decisions

- _System A latency typical: ..._
- _Did GPT estimate sensible numbers? Often null is OK._

---

# §3 — System B: LLM + ML (function calling to per-category models)

**Why**: this is the experimental treatment. Same LLM as System A, but with access
to a `predict_sales` tool that loads the right per-category pickle and returns
the prediction + SHAP drivers. The LLM uses this real number as ground truth
to write its reasoning + recommendations.

In [ ]:
# §3 v4 — L3 system: LLM + tuned ML + confidence layer (refactored from System B)

SYSTEM_L3_PROMPT = """You are Prometheus, an AI Virtual Colleague at Royal Unibrew specializing in beverage sales forecasting for the Danish market.

You have access to ONE tool: `predict_sales_with_confidence(brand, channel, date, category)`.
The tool returns: ML forecast (Optuna-tuned XGBoost/LightGBM), top-3 SHAP drivers, confidence (HIGH/LOW), confidence warnings, and abstain_recommended flag.

CATEGORY ROUTING (most common error source):
- If the user query specifies a category in parentheses (e.g., "MORENA (totalbeer brand)"), USE THAT CATEGORY DIRECTLY without checking against any internal mapping. The user knows their brand portfolio.
- The list below is NON-EXHAUSTIVE — it gives examples only. NEVER refuse to predict because a brand "is not in the list".

EXAMPLE BRANDS (non-exhaustive):
- csd: COCA COLA, PEPSI, FANTA, SPRITE, MIRINDA, FAXE KONDI, HARBOE-csd, ØRBÆK, SFC, THY
- danskvand: AQUA D'OR, EGEKILDE, RAMLOSA, BLUE KELD, KILDEVÆLD, SAN PELLEGRINO
- energidrikke: RED BULL, MONSTER, FAXE KONDI BOOSTER, CULT, STATE, NOCCO, POWERADE, GATORADE, VITAMIN WELL, AYYO, PRIME
- rtd: BREEZER, SOMERSBY, SHAKER, SMIRNOFF ICE, REKORDERLIG, BUZZBALLZ
- totalbeer: CARLSBERG, TUBORG, ROYAL, TO ØL, MIKKELLER, HOEGAARDEN, HARBOE-beer, ALBANI, ST.BERNARDUS, SAN MIGUEL, SAMUEL SMITH, EDELMEISTER, NASTRO AZZURRO, PAULANER, WIIBROE, MORENA, NØRREBRO, NYBORG DISTILLERY, WARWIK

VALID CHANNELS (use ONLY these 18, with underscores, UPPERCASE):
SUPERBRUGSEN, KVICKLY, BRUGSEN, MENY, SPAR, MIN_KOBMAND, FOTEX, BILKA, NETTO, REMA_1000, LOVBJERG, NEMLIG_COM, 7_ELEVEN, OK_PLUS, CIRCLE_K, Q8, BFI_SHELL, BFI_EXTRA

Do NOT invent channels (no "ON_PREMISE", "ON_TRADE", "ONLINE", "RETAIL", "SUPERMARKET", "ALL_CHANNELS").

PROCEDURE:
1. Identify the category (use category-in-parentheses from query if present)
2. Identify channel(s) — must be in VALID CHANNELS list
3. Call predict_sales_with_confidence(brand, channel, date, category)
4. FALLBACK: if tool returns "error" / "no data" for the requested channel, try at most 2 OTHER valid channels from the list before giving up. Mention in reasoning which channels were tried.
5. If `abstain_recommended` is True (LOW confidence): set predicted_sales_units to null AND explain the warnings in reasoning. Do not present a forecast you don't trust.
6. For multi-channel/multi-brand archetypes (top_performers, channel_ranking, channel_volatility, brand_vs_brand): you MAY call the tool up to 5 times — once per (brand, channel) tuple needed.

RESPONSE SCHEMA (strict JSON):
{
  "predicted_sales_units": <single number, or null if abstain. NEVER a dict, list, or string.>,
  "category_chosen": "<csd | danskvand | energidrikke | rtd | totalbeer>",
  "confidence": "HIGH" or "LOW",
  "confidence_warnings": [<list of warning strings, empty if HIGH>],
  "reasoning": "<grounded reasoning: cite top SHAP drivers AND the expected MAPE>",
  "drivers": [{"feature": "<name>", "shap": <number>, "value": <number>}, ...]
}

ARCHETYPE-SPECIFIC GUIDANCE:
- point_forecast / range_forecast / confidence_interval: predicted_sales_units = the single tool prediction. For CI/range, derive bounds = pred ± 1.28 × MAPE × pred (80% interval). For range_forecast use pred ± MAPE × pred.
- top_performers / brand_vs_brand: predicted_sales_units = the FIRST/focal brand's forecast (single number). Discuss the comparison/ranking in reasoning, listing each brand's predicted units.
- channel_ranking / channel_volatility: predicted_sales_units = the prediction for the FIRST/focal channel. Discuss ranking/variance in reasoning.
- driver_question: predicted_sales_units = the tool's forecast. Drivers field MUST contain top-3 from SHAP.
- anomaly_flag / yoy_comparison: predicted_sales_units = forecast value. Reasoning must compare to historical baseline mentioned in the query.

CRITICAL RULES:
- DO NOT invent marketing recommendations (no DKK budgets, no ROI, no % uplift speculation)
- DO NOT invent brand names not present in the user query
- DO NOT call the tool with channels outside VALID CHANNELS
- BE TRANSPARENT about confidence: never claim HIGH when tool says LOW
- predicted_sales_units MUST be a single scalar (number or null). If you have multiple brands/channels, pick the focal one and put others in reasoning.

ACTIONABLE INSIGHTS (in reasoning, kept short):
At the end of reasoning, provide 1-2 grounded actionable insights based ONLY on the data:
- "Driver lag_12 is dominant (+0.42 SHAP) → maintain current cadence at this channel"
- "Confidence LOW (only 12 train months) → recommend gathering more historical data before relying on this forecast"
- "Predicted -25% vs YoY → recommend investigating competitor activity at [channel] in [month]"

Allowed: directional advice grounded in SHAP/data. NOT allowed: DKK budgets, ROI multipliers, % uplift estimates, marketing campaign cost projections."""

PREDICT_TOOL_L3 = {
    "type": "function",
    "function": {
        "name": "predict_sales_with_confidence",
        "description": "Get the TUNED ML-forecasted sales (XGBoost/LightGBM with Optuna 50-trial tuning) for a (brand, channel, date) in a specific category. Returns predicted units + SHAP top-3 drivers + confidence label (HIGH/LOW) + warnings + abstain_recommended flag.",
        "parameters": {
            "type": "object",
            "properties": {
                "brand": {"type": "string", "description": "Brand UPPERCASE"},
                "channel": {"type": "string", "description": "Channel UPPERCASE with underscores"},
                "date": {"type": "string", "description": "YYYY-MM-01"},
                "category": {"type": "string", "enum": ["csd", "danskvand", "energidrikke", "rtd", "totalbeer"]},
            },
            "required": ["brand", "channel", "date", "category"],
        },
    },
}


def system_L3(query: str) -> dict:
    t0 = time.perf_counter()
    messages = [
        {"role": "system", "content": SYSTEM_L3_PROMPT},
        {"role": "user", "content": query},
    ]
    total_in, total_out, tool_calls_made, iteration = 0, 0, 0, 0
    max_iter = 3

    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages, tools=[PREDICT_TOOL_L3],
        tool_choice="auto", max_tokens=6000,
    )
    total_in += response.usage.prompt_tokens
    total_out += response.usage.completion_tokens
    msg = response.choices[0].message
    messages.append({"role": "assistant", "content": msg.content,
                     "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    while msg.tool_calls and iteration < max_iter:
        iteration += 1
        for tc in msg.tool_calls:
            tool_calls_made += 1
            args = json.loads(tc.function.arguments)
            result = predict_sales_with_confidence(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=[PREDICT_TOOL_L3],
            tool_choice="auto", response_format={"type": "json_object"}, max_tokens=6000,
        )
        total_in += response.usage.prompt_tokens
        total_out += response.usage.completion_tokens
        msg = response.choices[0].message
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    latency = time.perf_counter() - t0
    cost = (total_in * PRICING[OPENAI_MODEL]["input"] +
            total_out * PRICING[OPENAI_MODEL]["output"]) / 1_000_000
    try:
        parsed = json.loads(msg.content)
    except Exception:
        parsed = {"predicted_sales_units": None, "reasoning": msg.content or "", "drivers": [],
                  "confidence": "LOW", "confidence_warnings": [], "category_chosen": ""}

    return {
        "system": "L3", "raw_text": msg.content,
        "predicted_sales_units": parsed.get("predicted_sales_units"),
        "category_chosen": parsed.get("category_chosen", ""),
        "confidence": parsed.get("confidence", "LOW"),
        "confidence_warnings": parsed.get("confidence_warnings", []),
        "reasoning": parsed.get("reasoning", ""),
        "recommendations": "",
        "drivers": parsed.get("drivers", []),
        "tool_calls_made": tool_calls_made, "latency_s": latency,
        "input_tokens": total_in, "output_tokens": total_out, "cost_usd": cost,
    }


print("L3 system defined (LLM + tuned ML + confidence layer). Run cell 13 for smoke test.")


In [ ]:
# §3-fix — predict_sales_impl + L3 confidence layer wrapper (predict_sales_with_confidence)
import shap

def predict_sales_impl(brand, channel, date, category):
    """Tuned-ML prediction (Optuna 50-trial). Returns raw pred + model_used + expected_mape."""
    art = load_model_for_category(category)
    channel_normalized = channel.replace(" ", "_").upper()
    brand_normalized = brand.upper()
    target_ts = pd.Timestamp(date)
    fm = art["fm"]
    row = fm[
        (fm["brand"].str.upper() == brand_normalized) &
        (fm["channel"].str.upper() == channel_normalized) &
        (fm["date"] == target_ts)
    ]
    if len(row) == 0:
        # Fallback: try without channel normalization
        row = fm[
            (fm["brand"].str.upper() == brand_normalized) &
            (fm["channel"] == channel) &
            (fm["date"] == target_ts)
        ]
    if len(row) == 0:
        avail_channels = sorted(fm["channel"].unique())
        return {
            "error": f"No data for ({brand}, {channel}, {date}) in {category}",
            "available_channels_for_this_category": avail_channels[:18],
        }

    X = art["pipe"].transform(row.head(1))
    pred = float(np.clip(np.expm1(art["model"].predict(X)[0]), 0, None))
    return {
        "predicted_sales_units": round(pred, 0),
        "model_used": art["model_type"],
        "expected_per_brand_mape_pct": art["mape"],
        "_X": X,        # for SHAP
        "_row": row,    # for confidence checks
    }


# ── Confidence layer wrapper for L3 ─────────────────────────────────────────

CONFIDENCE_THRESHOLDS = {
    "max_mape_for_high": 30.0,    # if expected MAPE > 30% → LOW
    "min_train_obs_for_high": 18,  # if < 18 training months for this brand → LOW
}

# SHAP explainer cache
_shap_explainer_cache = {}
def _get_shap_explainer(category):
    if category not in _shap_explainer_cache:
        art = load_model_for_category(category)
        _shap_explainer_cache[category] = shap.TreeExplainer(art["model"])
    return _shap_explainer_cache[category]


def predict_sales_with_confidence(brand, channel, date, category):
    """L3 prediction with confidence layer + SHAP top-3 drivers."""
    raw = predict_sales_impl(brand, channel, date, category)
    if "error" in raw:
        return {**raw, "confidence": "LOW", "confidence_warnings": ["No data"], "abstain_recommended": True}

    art = load_model_for_category(category)
    fm = art["fm"]
    brand_n = brand.upper()
    n_train_obs = int(((fm["brand"].str.upper() == brand_n) & (fm["split"] == "train")).sum())

    # Confidence checks
    confidence = "HIGH"
    warnings = []
    if raw["expected_per_brand_mape_pct"] > CONFIDENCE_THRESHOLDS["max_mape_for_high"]:
        confidence = "LOW"
        warnings.append(f"Per-brand MAPE {raw['expected_per_brand_mape_pct']:.1f}% > {CONFIDENCE_THRESHOLDS['max_mape_for_high']}% threshold")
    if n_train_obs < CONFIDENCE_THRESHOLDS["min_train_obs_for_high"]:
        confidence = "LOW"
        warnings.append(f"Only {n_train_obs} training months for brand (< {CONFIDENCE_THRESHOLDS['min_train_obs_for_high']} required)")

    # SHAP top-3 drivers
    drivers = []
    try:
        X = raw.pop("_X")
        row = raw.pop("_row")
        explainer = _get_shap_explainer(category)
        shap_vals_raw = explainer.shap_values(X)
        if isinstance(shap_vals_raw, list):
            shap_vals = shap_vals_raw[0]
        elif shap_vals_raw.ndim > 1:
            shap_vals = shap_vals_raw[0]
        else:
            shap_vals = shap_vals_raw
        try:
            feat_names = list(art["pipe"].get_feature_names_out())
        except Exception:
            feat_names = [f"f_{i}" for i in range(X.shape[1])]
        top3 = sorted(zip(feat_names, shap_vals, X[0]), key=lambda t: abs(t[1]), reverse=True)[:3]
        drivers = [{"feature": str(f), "shap": round(float(s), 4), "value": round(float(v), 4)} for f, s, v in top3]
    except Exception as e:
        # Cleanup if SHAP fails
        raw.pop("_X", None)
        raw.pop("_row", None)
        drivers = []

    return {
        "predicted_sales_units": raw["predicted_sales_units"],
        "model_used": raw["model_used"],
        "expected_per_brand_mape_pct": raw["expected_per_brand_mape_pct"],
        "confidence": confidence,
        "confidence_warnings": warnings,
        "abstain_recommended": (confidence == "LOW"),
        "drivers": drivers,
        "n_train_observations_for_brand": n_train_obs,
    }


# Smoke tests
print("predict_sales_with_confidence smoke tests:\n")

print("[1] HIGH-confidence brand (CARLSBERG totalbeer):")
r = predict_sales_with_confidence("CARLSBERG", "MENY", "2025-10-01", "totalbeer")
print(f"  pred: {r.get('predicted_sales_units')}  confidence: {r.get('confidence')}  warnings: {r.get('confidence_warnings')}")
print(f"  drivers: {r.get('drivers', [])[:2]}\n")

print("[2] HIGH-confidence brand (COCA COLA csd):")
r = predict_sales_with_confidence("COCA COLA", "BILKA", "2025-10-01", "csd")
print(f"  pred: {r.get('predicted_sales_units')}  confidence: {r.get('confidence')}  warnings: {r.get('confidence_warnings')}")
print(f"  drivers: {r.get('drivers', [])[:2]}\n")

print("[3] L3 end-to-end smoke test:")
test = system_L3("Predict the monthly sales (units) for CARLSBERG at MENY in October 2025.")
print(f"  category: {test.get('category_chosen')}")
print(f"  predicted: {test['predicted_sales_units']}  confidence: {test.get('confidence')}")
print(f"  warnings: {test.get('confidence_warnings')}")
print(f"  reasoning: {test['reasoning'][:200]}...")
print(f"  drivers: {test.get('drivers', [])[:2]}")
print(f"  cost: ${test['cost_usd']:.4f}")


---

# §3.5 — L1 (raw data) + L2 (untuned ML) systems (Phase 5 — added 2026-04-26)

**Why**: Phase 4's A vs B comparison is a strawman (LLM-only vs LLM+tuned-ML).
To isolate the contribution of training quality, Phase 5 introduces 4 tiers:

| Tier | What | Tool |
|------|------|------|
| **L0** | LLM-only (current System A) | none |
| **L1** | LLM + raw historical data | `get_historical_sales(brand, channel, lookback_months=12)` |
| **L2** | LLM + untuned ML (n_est=200, no Optuna) | `predict_sales_untuned(...)` routes to `outputs_<cat>_untuned/` |
| **L3** | LLM + tuned ML + confidence layer (refactored from System B) | `predict_sales_with_confidence(...)` |

L1 tests "does data access alone help?" — the LLM has to extrapolate from numbers.
L2 tests "does ML structure help vs raw data?" — same architecture as L3 but no tuning.


In [ ]:
# §3.5a — L1 system: LLM + raw historical sales (NO ML, no leakage)
import time as _time

def get_historical_sales(brand: str, channel: str, target_date: str, lookback_months: int = 12, category: str | None = None) -> dict:
    """Return last N months of actual sales for (brand, channel) STRICTLY BEFORE target_date.

    The target_date filter prevents test-set leakage: when the LLM asks about a future month,
    it only sees historical data, never the ground-truth value.
    """
    brand_n = brand.upper()
    channel_n = channel.replace(" ", "_").upper()
    try:
        target_ts = pd.Timestamp(target_date)
    except Exception:
        return {"error": f"Cannot parse target_date {target_date!r}"}

    cats_to_search = [category] if category else list(CATEGORY_MODELS.keys())
    history = []
    for cat in cats_to_search:
        try:
            art = load_model_for_category(cat)
            fm = art["fm"]
            rows = fm[(fm["brand"].str.upper() == brand_n) &
                      (fm["channel"].str.upper() == channel_n) &
                      (fm["date"] < target_ts)]  # ← KEY FIX: strictly before target
            if len(rows) > 0:
                rows = rows.sort_values("date").tail(lookback_months)
                history = [{"date": d.strftime("%Y-%m-01"), "sales_units": float(s)}
                           for d, s in zip(rows["date"], rows["sales_units"])]
                break
        except Exception:
            continue

    if not history:
        return {"error": f"No historical data found for ({brand!r}, {channel!r}) before {target_date}"}

    return {
        "brand": brand, "channel": channel, "target_date": target_date,
        "history": history,
        "n_months_returned": len(history),
        "earliest_date": history[0]["date"],
        "latest_date": history[-1]["date"],
    }


HISTORY_TOOL = {
    "type": "function",
    "function": {
        "name": "get_historical_sales",
        "description": "Get last N months of ACTUAL sales (no ML prediction) for a (brand, channel) pair, STRICTLY BEFORE the target_date. Use this to extrapolate trends and seasonality manually for the future target month.",
        "parameters": {
            "type": "object",
            "properties": {
                "brand": {"type": "string", "description": "Brand name UPPERCASE"},
                "channel": {"type": "string", "description": "Channel name UPPERCASE with underscores"},
                "target_date": {"type": "string", "description": "The future target date YYYY-MM-01 — only history strictly before this date is returned"},
                "lookback_months": {"type": "integer", "description": "Months of history to return (default 12)"},
            },
            "required": ["brand", "channel", "target_date"],
        },
    },
}


SYSTEM_L1_PROMPT = """You are Prometheus, an AI Virtual Colleague at Royal Unibrew.

You have access to ONE tool: `get_historical_sales(brand, channel, target_date, lookback_months=12)` which returns the last N months of ACTUAL sales for a (brand, channel) pair, STRICTLY BEFORE the target_date.

You DO NOT have access to any forecasting model. You DO NOT see the target month's actual value. You must extrapolate from the historical numbers using:
- Seasonal patterns (e.g. beer peaks in summer)
- Trends (growth or decline over the last 12 months)
- Year-over-year comparison (compare same month last year if available)

VALID CHANNELS (use ONLY these 18, with underscores, UPPERCASE):
SUPERBRUGSEN, KVICKLY, BRUGSEN, MENY, SPAR, MIN_KOBMAND, FOTEX, BILKA, NETTO, REMA_1000, LOVBJERG, NEMLIG_COM, 7_ELEVEN, OK_PLUS, CIRCLE_K, Q8, BFI_SHELL, BFI_EXTRA

Do NOT invent channels.

PROCEDURE:
1. Identify (brand, channel, target_date) from the query — channel MUST be in VALID CHANNELS
2. Call get_historical_sales ONCE with the future target_date — up to 5 times for multi-channel/multi-brand prompts
3. FALLBACK: if tool returns "error" / "no historical data" for the requested channel, try at most 2 OTHER valid channels. Mention which channels were tried.
4. Use the returned numbers to estimate the forecast — be honest: without ML, your forecast is a heuristic extrapolation

RESPONSE SCHEMA (strict JSON):
{
  "predicted_sales_units": <single number from extrapolation, or null. NEVER a dict, list, or string.>,
  "category_chosen": "",
  "confidence": "LOW",
  "reasoning": "<explain the historical pattern (cite specific past months) and how you derived the estimate; for multi-brand prompts list per-brand forecast in reasoning>",
  "drivers": []
}

CRITICAL RULES:
- DO NOT invent brand names not in the user query
- DO NOT call the tool with channels outside VALID CHANNELS
- predicted_sales_units MUST be a single scalar — pick focal brand/channel
- confidence is always LOW (no ML model)"""

def system_L1(query: str) -> dict:
    t0 = _time.perf_counter()
    messages = [
        {"role": "system", "content": SYSTEM_L1_PROMPT},
        {"role": "user", "content": query},
    ]
    total_in, total_out, tool_calls_made, iteration = 0, 0, 0, 0
    max_iter = 3

    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages, tools=[HISTORY_TOOL],
        tool_choice="auto", max_tokens=6000,
    )
    total_in += response.usage.prompt_tokens
    total_out += response.usage.completion_tokens
    msg = response.choices[0].message
    messages.append({"role": "assistant", "content": msg.content,
                     "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    while msg.tool_calls and iteration < max_iter:
        iteration += 1
        for tc in msg.tool_calls:
            tool_calls_made += 1
            args = json.loads(tc.function.arguments)
            result = get_historical_sales(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=[HISTORY_TOOL],
            tool_choice="auto", response_format={"type": "json_object"}, max_tokens=6000,
        )
        total_in += response.usage.prompt_tokens
        total_out += response.usage.completion_tokens
        msg = response.choices[0].message
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    latency = _time.perf_counter() - t0
    cost = (total_in * PRICING[OPENAI_MODEL]["input"] +
            total_out * PRICING[OPENAI_MODEL]["output"]) / 1_000_000
    try:
        parsed = json.loads(msg.content)
    except Exception:
        parsed = {"predicted_sales_units": None, "reasoning": msg.content or "", "drivers": [], "confidence": "LOW", "category_chosen": ""}

    return {
        "system": "L1", "raw_text": msg.content,
        "predicted_sales_units": parsed.get("predicted_sales_units"),
        "category_chosen": parsed.get("category_chosen", ""),
        "confidence": parsed.get("confidence", "LOW"),
        "reasoning": parsed.get("reasoning", ""),
        "recommendations": "",
        "drivers": parsed.get("drivers", []),
        "tool_calls_made": tool_calls_made, "latency_s": latency,
        "input_tokens": total_in, "output_tokens": total_out, "cost_usd": cost,
    }


# Smoke test — verify NO leakage (target Oct 2025 should not appear in history)
print("L1 system FIXED. Smoke test (verify no leakage):")
hist = get_historical_sales("CARLSBERG", "MENY", target_date="2025-10-01", lookback_months=12)
if "history" in hist:
    print(f"  Returned {len(hist['history'])} months: {hist['earliest_date']} → {hist['latest_date']}")
    assert all(h["date"] < "2025-10-01" for h in hist["history"]), "LEAKAGE DETECTED"
    print("  ✓ No leakage — all dates < target")
else:
    print(f"  {hist}")

test = system_L1("Predict the monthly sales (units) for CARLSBERG at MENY in October 2025.")
print(f"  predicted: {test['predicted_sales_units']}")
print(f"  confidence: {test['confidence']}")
print(f"  reasoning[:200]: {test['reasoning'][:200]}...")
print(f"  cost: ${test['cost_usd']:.4f}")


In [ ]:
# §3.5b — L2 system: LLM + untuned ML (no confidence layer)

# Per-category UNTUNED metadata (from outputs_<cat>_untuned/metrics_untuned.json)
import json as _json

CATEGORY_MODELS_UNTUNED = {
    "csd":          {"out": ANALYSIS_DIR / "outputs_csd_untuned"},
    "danskvand":    {"out": ANALYSIS_DIR / "outputs_danskvand_untuned"},
    "energidrikke": {"out": ANALYSIS_DIR / "outputs_energidrikke_untuned"},
    "rtd":          {"out": ANALYSIS_DIR / "outputs_rtd_untuned"},
    "totalbeer":    {"out": ANALYSIS_DIR / "outputs_totalbeer_untuned"},
}
# Load metrics_untuned.json for each
for cat, cfg in CATEGORY_MODELS_UNTUNED.items():
    with open(cfg["out"] / "metrics_untuned.json") as f:
        m = _json.load(f)
    cfg["primary"] = m["primary_model"]
    _key = "xgb" if cfg["primary"] == "xgboost" else "lgb"
    cfg["mape"] = m[_key]["per_brand_median_mape_pct"]
    cfg["fm"] = "feature_matrix_split.parquet"

_model_cache_untuned = {}
def load_model_for_category_untuned(cat):
    if cat in _model_cache_untuned: return _model_cache_untuned[cat]
    cfg = CATEGORY_MODELS_UNTUNED[cat]
    with open(cfg["out"] / "pipelines" / f"model_{cfg['primary']}.pkl", "rb") as f:
        model = pickle.load(f)
    with open(cfg["out"] / "pipelines" / "pipe_tree.pkl", "rb") as f:
        pipe = pickle.load(f)
    fm = pd.read_parquet(cfg["out"] / cfg["fm"])
    art = {"model": model, "pipe": pipe, "fm": fm, "model_type": cfg["primary"], "mape": cfg["mape"]}
    _model_cache_untuned[cat] = art
    return art


def predict_sales_untuned_impl(brand, channel, date, category):
    """L2 prediction: untuned model, no confidence check."""
    art = load_model_for_category_untuned(category)
    channel_n = channel.replace(" ", "_").upper()
    brand_n = brand.upper()
    target_ts = pd.Timestamp(date)
    fm = art["fm"]
    row = fm[(fm["brand"].str.upper() == brand_n) &
             (fm["channel"].str.upper() == channel_n) &
             (fm["date"] == target_ts)]
    if len(row) == 0:
        return {"error": f"No data for ({brand}, {channel}, {date}) in {category}"}
    X = art["pipe"].transform(row.head(1))
    pred = float(np.clip(np.expm1(art["model"].predict(X)[0]), 0, None))
    return {
        "predicted_sales_units": round(pred, 0),
        "model_used": art["model_type"],
        "expected_per_brand_mape_pct": art["mape"],
        "training_quality": "untuned (n_est=200, default params)",
    }


PREDICT_TOOL_UNTUNED = {
    "type": "function",
    "function": {
        "name": "predict_sales_untuned",
        "description": "Get the UNTUNED ML-forecasted sales for (brand, channel, date) in a category. Returns predicted units + expected MAPE. Models trained with default params (no Optuna).",
        "parameters": {
            "type": "object",
            "properties": {
                "brand": {"type": "string", "description": "Brand UPPERCASE"},
                "channel": {"type": "string", "description": "Channel UPPERCASE with underscores"},
                "date": {"type": "string", "description": "YYYY-MM-01"},
                "category": {"type": "string", "enum": ["csd", "danskvand", "energidrikke", "rtd", "totalbeer"]},
            },
            "required": ["brand", "channel", "date", "category"],
        },
    },
}


SYSTEM_L2_PROMPT = """You are Prometheus, an AI Virtual Colleague at Royal Unibrew specializing in beverage sales forecasting for the Danish market.

You have access to ONE tool: `predict_sales_untuned(brand, channel, date, category)`.
The tool routes to UNTUNED ML models (default hyperparameters, no Optuna). Expected per-brand MAPE is 22-30%.

CATEGORY ROUTING (most common error source):
- If the user query specifies a category in parentheses (e.g., "MORENA (totalbeer brand)"), USE THAT CATEGORY DIRECTLY without checking against any internal mapping.
- The list below is NON-EXHAUSTIVE. NEVER refuse to predict because a brand "is not in the list".

EXAMPLE BRANDS (non-exhaustive):
- csd: COCA COLA, PEPSI, FANTA, SPRITE, MIRINDA, FAXE KONDI, HARBOE-csd, ØRBÆK, SFC, THY
- danskvand: AQUA D'OR, EGEKILDE, RAMLOSA, BLUE KELD, KILDEVÆLD, SAN PELLEGRINO
- energidrikke: RED BULL, MONSTER, FAXE KONDI BOOSTER, CULT, STATE, NOCCO, POWERADE, GATORADE, VITAMIN WELL, AYYO, PRIME
- rtd: BREEZER, SOMERSBY, SHAKER, SMIRNOFF ICE, REKORDERLIG, BUZZBALLZ
- totalbeer: CARLSBERG, TUBORG, ROYAL, TO ØL, MIKKELLER, HOEGAARDEN, HARBOE-beer, ALBANI, ST.BERNARDUS, SAN MIGUEL, SAMUEL SMITH, EDELMEISTER, NASTRO AZZURRO, PAULANER, WIIBROE, MORENA, NØRREBRO, NYBORG DISTILLERY, WARWIK

VALID CHANNELS (use ONLY these 18, with underscores, UPPERCASE):
SUPERBRUGSEN, KVICKLY, BRUGSEN, MENY, SPAR, MIN_KOBMAND, FOTEX, BILKA, NETTO, REMA_1000, LOVBJERG, NEMLIG_COM, 7_ELEVEN, OK_PLUS, CIRCLE_K, Q8, BFI_SHELL, BFI_EXTRA

Do NOT invent channels.

PROCEDURE:
1. Identify the category (use category-in-parentheses from query if present)
2. Call predict_sales_untuned(brand, channel, date, category) — once for single-brand, up to 5 times for multi-brand/multi-channel
3. FALLBACK: if tool returns "error" / "no data" for the requested channel, try at most 2 OTHER valid channels before giving up. Mention which channels were tried.
4. Be transparent about the untuned nature (~25-30% expected MAPE)

RESPONSE SCHEMA (strict JSON):
{
  "predicted_sales_units": <single number, or null. NEVER a dict, list, or string.>,
  "category_chosen": "<cat>",
  "confidence": "MEDIUM",
  "reasoning": "<grounded in untuned ML prediction; mention MAPE; for multi-brand prompts list each brand's forecast in reasoning>",
  "drivers": []
}

CRITICAL RULES:
- DO NOT invent marketing recommendations
- DO NOT invent brand names not in the user query
- DO NOT call the tool with channels outside VALID CHANNELS
- predicted_sales_units MUST be a single scalar — if multi-brand, pick the focal brand and put others in reasoning"""

def system_L2(query: str) -> dict:
    t0 = _time.perf_counter()
    messages = [
        {"role": "system", "content": SYSTEM_L2_PROMPT},
        {"role": "user", "content": query},
    ]
    total_in, total_out, tool_calls_made, iteration = 0, 0, 0, 0
    max_iter = 3

    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages, tools=[PREDICT_TOOL_UNTUNED],
        tool_choice="auto", max_tokens=6000,
    )
    total_in += response.usage.prompt_tokens
    total_out += response.usage.completion_tokens
    msg = response.choices[0].message
    messages.append({"role": "assistant", "content": msg.content,
                     "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    while msg.tool_calls and iteration < max_iter:
        iteration += 1
        for tc in msg.tool_calls:
            tool_calls_made += 1
            args = json.loads(tc.function.arguments)
            result = predict_sales_untuned_impl(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=[PREDICT_TOOL_UNTUNED],
            tool_choice="auto", response_format={"type": "json_object"}, max_tokens=6000,
        )
        total_in += response.usage.prompt_tokens
        total_out += response.usage.completion_tokens
        msg = response.choices[0].message
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [t.model_dump() for t in (msg.tool_calls or [])]})

    latency = _time.perf_counter() - t0
    cost = (total_in * PRICING[OPENAI_MODEL]["input"] +
            total_out * PRICING[OPENAI_MODEL]["output"]) / 1_000_000
    try:
        parsed = json.loads(msg.content)
    except Exception:
        parsed = {"predicted_sales_units": None, "reasoning": msg.content or "", "drivers": [], "confidence": "MEDIUM", "category_chosen": ""}

    return {
        "system": "L2", "raw_text": msg.content,
        "predicted_sales_units": parsed.get("predicted_sales_units"),
        "category_chosen": parsed.get("category_chosen", ""),
        "confidence": parsed.get("confidence", "MEDIUM"),
        "reasoning": parsed.get("reasoning", ""),
        "recommendations": "",
        "drivers": parsed.get("drivers", []),
        "tool_calls_made": tool_calls_made, "latency_s": latency,
        "input_tokens": total_in, "output_tokens": total_out, "cost_usd": cost,
    }


# Smoke test
print("L2 system defined. Smoke test:")
test = system_L2("Predict the monthly sales (units) for CARLSBERG at MENY in October 2025.")
print(f"  category: {test['category_chosen']}")
print(f"  predicted: {test['predicted_sales_units']}")
print(f"  confidence: {test['confidence']}")
print(f"  reasoning: {test['reasoning'][:200]}...")
print(f"  cost: ${test['cost_usd']:.4f}")


### §3 — Observations + Decisions

- _Did B make a tool call? (should always be 1+)_
- _B's predicted number ≈ ML model prediction?_
- _Latency overhead vs A: ..._

---

# §4 — System C: Prometheus (placeholder, enabled when API ready)

**Why**: when Manifold provides API access to Prometheus (in 3-5 days), this cell
will be enabled and `system_C` will become a third datapoint in the comparison.

For now, leave `PROMETHEUS_API_AVAILABLE = False` — the run loop in §5 will skip C.

**When Prometheus arrives**:
1. Get API key + endpoint from Manifold
2. Add `PROMETHEUS_API_KEY=...` and `PROMETHEUS_ENDPOINT=...` to `.env`
3. Set `PROMETHEUS_API_AVAILABLE = True` in this cell
4. Re-run §4 and §5 (200 prompts × 3 systems = 150 calls, ~10 min)

In [6]:
# §4 — System C placeholder
PROMETHEUS_API_AVAILABLE = False  # Set True when API is ready

def system_C(query: str) -> dict:
    """Placeholder for Prometheus API call. Enable when Manifold provides access."""
    if not PROMETHEUS_API_AVAILABLE:
        return None  # signals "skip this system"

    import requests
    api_key = os.environ.get("PROMETHEUS_API_KEY")
    endpoint = os.environ.get("PROMETHEUS_ENDPOINT")
    if not api_key or not endpoint:
        return None

    t0 = time.perf_counter()
    response = requests.post(
        endpoint,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"query": query},
        timeout=60,
    )
    latency = time.perf_counter() - t0
    response.raise_for_status()
    data = response.json()

    # Parse response — adjust based on Prometheus actual API schema
    return {
        "system": "C",
        "raw_text": data.get("response", str(data)),
        "predicted_sales_units": data.get("forecast"),
        "reasoning": data.get("reasoning", ""),
        "recommendations": data.get("recommendations", ""),
        "latency_s": latency,
        "input_tokens": data.get("input_tokens", 0),
        "output_tokens": data.get("output_tokens", 0),
        "cost_usd": data.get("cost_usd", 0.0),
    }

print(f"PROMETHEUS_API_AVAILABLE = {PROMETHEUS_API_AVAILABLE}")
if PROMETHEUS_API_AVAILABLE:
    print("System C will be included in §5 run loop.")
else:
    print("System C is DISABLED. Re-enable when Manifold gives API access.")


PROMETHEUS_API_AVAILABLE = False
System C is DISABLED. Re-enable when Manifold gives API access.


### §4 — Observations + Decisions

- Prometheus access status: _waiting / received_
- API endpoint + key in .env: _no / yes_

---

# §5 — Run all 50 prompts × all available systems

**Why**: this is the data-collection step. Saves results to disk after each prompt
so you don't lose progress on interrupt.

**Cost estimate** (with C disabled): 50 × 2 systems × ~$0.001 = **~$0.10**
**Cost estimate** (with C enabled): 50 × 3 systems × ~$0.001 = **~$0.15** (Prometheus cost unknown)

**Time estimate**: ~5 min for A+B; +~3 min if C enabled.

In [ ]:
# §5 v4 — Phase 5: Run BOTH 4 systems (L0/L1/L2/L3) on 50 prompts (200 calls total)
import time as _time

# Map L0 → system_A (legacy name retained), L1 → system_L1, L2 → system_L2, L3 → system_L3
SYSTEMS = [
    ("L0", system_A),
    ("L1", system_L1),
    ("L2", system_L2),
    ("L3", system_L3),
]

all_responses = []  # fresh start
results_path = OUTPUT_DIR / "raw_responses.parquet"
t_start = _time.perf_counter()

def _save_incremental():
    df_save = pd.DataFrame(all_responses)
    for col in df_save.select_dtypes(include=["object"]).columns:
        df_save[col] = df_save[col].apply(lambda v: str(v) if isinstance(v, (dict, list)) else v)
        df_save[col] = df_save[col].astype(str).replace("None", "")
    df_save["predicted_sales_units"] = pd.to_numeric(df_save["predicted_sales_units"], errors="coerce")
    df_save.to_parquet(results_path, index=False)

for system_name, system_fn in SYSTEMS:
    print(f"\n{'='*70}\n  SYSTEM {system_name}\n{'='*70}")
    for _, prompt in prompts_df.iterrows():
        if any(r["prompt_id"] == prompt["id"] and r["system"] == system_name for r in all_responses):
            continue

        for attempt in range(3):
            try:
                result = system_fn(prompt["query"])
                # Normalize system label (system_A returns "A", overwrite to "L0")
                result["system"] = system_name
                result.update({
                    "prompt_id": prompt["id"],
                    "category": prompt["category"],
                    "archetype": prompt["archetype"],
                    "query": prompt["query"],
                    "actual_sales_units": prompt["actual_sales_units"],
                })
                all_responses.append(result)
                _save_incremental()

                cat = result.get("category_chosen", "")[:14]
                pred = result["predicted_sales_units"]
                conf = result.get("confidence", "")[:6]
                print(f"  ✓ p{prompt['id']:>2d} sys={system_name}  cat={cat:<14s}  pred={pred!s:<10s}  conf={conf:<6s}  actual={prompt['actual_sales_units']:>10,.0f}  lat={result['latency_s']:.1f}s")
                break
            except Exception as e:
                err = str(e)[:80]
                if "429" in err or "rate" in err.lower():
                    wait = 30 * (attempt + 1)
                    print(f"  ⏸ p{prompt['id']} sys={system_name} rate-limited, sleeping {wait}s ({err})")
                    _time.sleep(wait)
                else:
                    print(f"  ✗ p{prompt['id']} sys={system_name} FAILED: {err}")
                    break

print(f"\n✅ Done. Total responses: {len(all_responses)}")
print(f"   Elapsed: {(_time.perf_counter() - t_start)/60:.1f} min")
print(f"   Saved: {results_path}")
print(f"\nBreakdown:")
df_done = pd.DataFrame(all_responses)
print(df_done.groupby("system").size().to_string())


In [9]:
# §5b — Backup safe save (versione bullet-proof)
import pandas as pd

df_backup = pd.DataFrame(all_responses)

# Converte TUTTE le colonne con tipi misti a stringa (parquet vuole tipi puliti)
for col in df_backup.select_dtypes(include=["object"]).columns:
    df_backup[col] = df_backup[col].apply(
        lambda v: str(v) if isinstance(v, (dict, list)) else v
    )
    # Force to string for safety
    df_backup[col] = df_backup[col].astype(str).replace("None", "")

# La colonna predicted può essere numero — la teniamo separata
df_backup["predicted_sales_units"] = pd.to_numeric(
    df_backup["predicted_sales_units"], errors="coerce"
)

df_backup.to_parquet(OUTPUT_DIR / "raw_responses_v1_backup.parquet", index=False)
print(f"✅ Backup salvato: {len(df_backup)} righe")


✅ Backup salvato: 95 righe


### §5 — Observations + Decisions

- _Total responses generated: ..._
- _Total cost: ..._
- _Average latency per system: ..._

---

# §6 — Quantitative auto metrics

**Why**: 5 forecast accuracy + 2 operational metrics. All computed from §5 responses.

| Metric | Formula |
|--------|---------|
| MAPE | median(\|y-ŷ\|/y × 100) |
| MAE | median(\|y-ŷ\|) |
| RMSE | √mean((y-ŷ)²) |
| sMAPE | median(2·\|y-ŷ\|/(\|y\|+\|ŷ\|) × 100) |
| Hallucination rate | % predictions outside historical range |
| Latency p50/p95 | median + 95th percentile |
| Cost | mean USD per query |

In [ ]:
# §6 v4 — Quantitative metrics for 4-tier (raw + actual>=100 filter to reduce small-actual noise)
df = pd.read_parquet(OUTPUT_DIR / "raw_responses.parquet")

df["pred_num"] = pd.to_numeric(df["predicted_sales_units"], errors="coerce")
df["actual_num"] = pd.to_numeric(df["actual_sales_units"], errors="coerce")

prompts_lookup = prompts_df.set_index("id")[["brand", "channel", "date"]].to_dict("index")
df["brand"] = df["prompt_id"].map(lambda i: prompts_lookup.get(i, {}).get("brand"))

def is_hallucination(r):
    if pd.isna(r["pred_num"]): return False
    if r["pred_num"] < 0: return True
    cat = r.get("category", "")
    if not cat or cat not in CATEGORY_MODELS: return False
    art = load_model_for_category(cat)
    train = art["fm"][(art["fm"]["brand"] == r.get("brand", "")) & (art["fm"]["split"] == "train")]
    if len(train) == 0: return False
    return r["pred_num"] > 5 * train["sales_units"].max()

def is_abstain(r):
    return pd.isna(r["pred_num"])

def compute_metrics(df_in, label, min_actual=0):
    rows = []
    df_eval = df_in[(df_in["actual_num"] >= max(min_actual, 1)) & (~df_in["pred_num"].isna())].copy()
    df_eval["abs_err"] = (df_eval["actual_num"] - df_eval["pred_num"]).abs()
    df_eval["sq_err"] = (df_eval["actual_num"] - df_eval["pred_num"])**2
    df_eval["ape"] = df_eval["abs_err"] / df_eval["actual_num"] * 100
    df_eval["sape"] = 2 * df_eval["abs_err"] / (df_eval["actual_num"].abs() + df_eval["pred_num"].abs()) * 100
    for sys_name in ["L0", "L1", "L2", "L3"]:
        g = df_in[df_in["system"] == sys_name]
        g_eval = df_eval[df_eval["system"] == sys_name]
        rows.append({
            "system": sys_name,
            "n_total": len(g),
            "n_eval": len(g_eval),
            "abstain_rate_pct": g.apply(is_abstain, axis=1).sum() / max(len(g), 1) * 100,
            "MAPE_median": g_eval["ape"].median() if len(g_eval) > 0 else np.nan,
            "sMAPE_median": g_eval["sape"].median() if len(g_eval) > 0 else np.nan,
            "MAE_median": g_eval["abs_err"].median() if len(g_eval) > 0 else np.nan,
            "hallucination_rate_pct": g.apply(is_hallucination, axis=1).mean() * 100,
            "latency_p50_s": g["latency_s"].median(),
            "cost_total_usd": g["cost_usd"].sum(),
        })
    quant_df = pd.DataFrame(rows)
    print(f"\n{'='*90}\n  {label}\n{'='*90}")
    print(quant_df.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    return quant_df, df_eval

# Run on RAW (all valid actuals)
quant_raw, df_eval_raw = compute_metrics(df, "RAW (all actuals >= 1)", min_actual=1)

# Run with min_actual >= 100 (removes small-volume noise)
quant_filt, df_eval_filt = compute_metrics(df, "FILTERED (actual >= 100 units)", min_actual=100)

quant_raw.to_csv(OUTPUT_DIR / "quantitative_metrics_raw.csv", index=False)
quant_filt.to_csv(OUTPUT_DIR / "quantitative_metrics_filtered.csv", index=False)

# Per-archetype breakdown (filtered to actual>=100, sMAPE for stability)
print(f"\n{'='*90}\n  Per-archetype median sMAPE (FILTERED actual>=100)\n{'='*90}")
arch_smape = df_eval_filt.groupby(["archetype", "system"])["sape"].median().unstack().round(2)
print(arch_smape.to_string(na_rep="—"))
arch_smape.to_csv(OUTPUT_DIR / "smape_by_archetype.csv")

# Per-category breakdown (filtered)
print(f"\n{'='*90}\n  Per-category median sMAPE (FILTERED actual>=100)\n{'='*90}")
cat_smape = df_eval_filt.groupby(["category", "system"])["sape"].median().unstack().round(2)
print(cat_smape.to_string(na_rep="—"))
cat_smape.to_csv(OUTPUT_DIR / "smape_by_category.csv")

# Volume distribution check
print(f"\n{'='*90}\n  Prompt actual_sales_units distribution\n{'='*90}")
print(f"  All prompts: {len(prompts_df)}")
print(f"  actual < 10:    {(prompts_df['actual_sales_units'] < 10).sum():>3d}")
print(f"  actual 10-99:   {((prompts_df['actual_sales_units'] >= 10) & (prompts_df['actual_sales_units'] < 100)).sum():>3d}")
print(f"  actual 100-999: {((prompts_df['actual_sales_units'] >= 100) & (prompts_df['actual_sales_units'] < 1000)).sum():>3d}")
print(f"  actual >= 1000: {(prompts_df['actual_sales_units'] >= 1000).sum():>3d}")


### §6 — Observations + Decisions

- _MAPE A vs B vs C: ..._
- _Hallucination rate per system: ..._
- _Cost per system: ..._

---

# §7 — Qualitative metrics via LLM-as-judge (Sonnet 4.6)

**Why**: 5 metrics × 50 prompts × N systems = 250-750 judge calls (~$5).
Each metric has a G-Eval style prompt asking Sonnet to score 1-5 with rationale.

In [40]:
# §7 — LLM-as-judge
JUDGE_PROMPTS = {
    "groundedness": """You evaluate beverage sales forecasting agent outputs. Score the response on **groundedness** from 1 (totally vague) to 5 (every claim is grounded in specific data).

GROUNDEDNESS = the response references specific numeric values, drivers, brand names, channels, dates, or historical patterns rather than generic statements.

Examples:
- 5: "Forecast: 12,345 units. Driven by lag_12 (+0.42 SHAP) showing strong year-on-year. SUPERBRUGSEN promo penetration was 18% last October."
- 3: "Sales should be moderate. Some seasonal effects expected."
- 1: "It's hard to predict. Many factors play a role."

Respond ONLY with JSON: {"score": <1-5>, "rationale": "<one sentence>"}""",

    "actionability": """You evaluate sales recommendations for Royal Unibrew brand managers. Score **actionability** from 1 (cannot act on this) to 5 (clear, immediate next step).

ACTIONABLE = recommendation specifies WHAT to do, ON WHICH lever, BY WHEN, with specific NUMBER.

Examples:
- 5: "Increase distribution by 3pp at MENY by November 1st, 2025 — promo intensity is at 22% above baseline."
- 3: "Consider increasing promotional activity."
- 1: "Monitor market trends."

Respond ONLY with JSON: {"score": <1-5>, "rationale": "<one sentence>"}""",

    "specificity": """Score **specificity** of the response from 1 (entirely generic) to 5 (every statement is specific).

SPECIFIC = mentions named entities (brand X), named retailers (NETTO), specific dates (October 2025), exact numbers (12,345 units, +5pp), specific actions.

Respond ONLY with JSON: {"score": <1-5>, "rationale": "<one sentence>"}""",

    "coherence": """Score **coherence** of the reasoning chain from 1 (contradictions, non-sequitur) to 5 (every step follows from the previous).

COHERENT = forecast → drivers → recommendation form a logical narrative without internal contradictions.

Respond ONLY with JSON: {"score": <1-5>, "rationale": "<one sentence>"}""",

    "business_relevance": """Score **business relevance** for a Royal Unibrew brand manager from 1 (irrelevant noise) to 5 (addresses a real business priority: margin, share, distribution, promo ROI).

Respond ONLY with JSON: {"score": <1-5>, "rationale": "<one sentence>"}""",
}

def judge_response(metric: str, query: str, response_text: str) -> dict:
    """Single-metric LLM-as-judge using Sonnet 4.6."""
    prompt = f"""QUERY:
{query}

RESPONSE TO EVALUATE:
{response_text[:2000]}"""
    msg = anthropic_client.messages.create(
        model=JUDGE_MODEL, max_tokens=200, temperature=0,
        system=JUDGE_PROMPTS[metric],
        messages=[{"role": "user", "content": prompt}],
    )
    raw = msg.content[0].text.strip()
    # Strip code fences
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
        raw = raw.strip()
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = {"score": 0, "rationale": raw[:200]}
    cost = (msg.usage.input_tokens * PRICING[JUDGE_MODEL]["input"] +
            msg.usage.output_tokens * PRICING[JUDGE_MODEL]["output"]) / 1_000_000
    return {**parsed, "judge_cost_usd": cost}


# Run judge for all metrics × all responses
judge_path = OUTPUT_DIR / "judge_scores.parquet"
if judge_path.exists():
    judge_df = pd.read_parquet(judge_path)
    print(f"Loaded {len(judge_df)} existing judge scores")
else:
    judge_df = pd.DataFrame()

done = set(zip(judge_df["prompt_id"], judge_df["system"], judge_df["metric"])) if len(judge_df) else set()
new_rows = []

for _, r in df.iterrows():
    full_text = (r.get("reasoning", "") or "") + "\n\n" + (r.get("recommendations", "") or "")
    for metric in JUDGE_PROMPTS:
        key = (r["prompt_id"], r["system"], metric)
        if key in done:
            continue
        try:
            res = judge_response(metric, r["query"], full_text)
            new_rows.append({
                "prompt_id": r["prompt_id"], "system": r["system"], "metric": metric,
                "score": res.get("score", 0), "rationale": res.get("rationale", ""),
                "judge_cost_usd": res["judge_cost_usd"],
            })
        except Exception as e:
            print(f"  ✗ prompt {r['prompt_id']} sys={r['system']} metric={metric}: {e}")

if new_rows:
    judge_df = pd.concat([judge_df, pd.DataFrame(new_rows)], ignore_index=True)
    judge_df.to_parquet(judge_path, index=False)

print(f"\n✅ Total judge scores: {len(judge_df)}")
print(f"Total judge cost: ${judge_df['judge_cost_usd'].sum():.2f}")

# Aggregate scores per (system, metric)
print("\nMean qualitative scores (1-5 scale):")
qual_pivot = judge_df.groupby(["system", "metric"])["score"].mean().unstack()
print(qual_pivot.to_string(float_format=lambda x: f"{x:.2f}"))
qual_pivot.to_csv(OUTPUT_DIR / "qualitative_scores_by_system.csv")



✅ Total judge scores: 500
Total judge cost: $1.36

Mean qualitative scores (1-5 scale):
metric  actionability  business_relevance  coherence  groundedness  specificity
system                                                                         
A                2.28                3.40       2.50          2.04         3.04
B                3.56                3.78       3.22          3.52         4.44


### §7 — Observations + Decisions

- _Top metric for B vs A: ..._
- _Weakest metric for any system: ..._
- _Total judge cost: ..._

---

# §8 — Pairwise preference (LLM judge)

**Why**: per Zheng et al. 2023, pairwise is more consistent than absolute rating.
For each prompt, ask Sonnet which output it prefers (A vs B; if C exists: A vs C, B vs C).

**Output**: win rate per system pair.

In [ ]:
# §8 — Pairwise preference
PAIRWISE_PROMPT = """You are evaluating two outputs from beverage sales forecasting agents responding to the same query for Royal Unibrew brand managers.

Choose which output you prefer for supporting an actual business decision. Consider: groundedness in data, actionability, specificity, coherence, business relevance.

Respond ONLY with JSON: {"winner": "1" or "2" or "tie", "rationale": "<one sentence>"}"""

def pairwise_judge(query, output1, output2):
    user_msg = f"""QUERY:
{query}

OUTPUT 1:
{output1[:1500]}

OUTPUT 2:
{output2[:1500]}"""
    msg = anthropic_client.messages.create(
        model=JUDGE_MODEL, max_tokens=200, temperature=0,
        system=PAIRWISE_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )
    raw = msg.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
        raw = raw.strip()
    try: return json.loads(raw)
    except Exception: return {"winner": "tie", "rationale": raw[:200]}

# Build pairs
systems_present = sorted(df["system"].unique())
pairs = [(s1, s2) for i, s1 in enumerate(systems_present) for s2 in systems_present[i+1:]]
print(f"Pairwise comparisons: {pairs}")

pairwise_path = OUTPUT_DIR / "pairwise.parquet"
if pairwise_path.exists():
    pw_df = pd.read_parquet(pairwise_path)
else:
    pw_df = pd.DataFrame()

done_pairs = set(zip(pw_df["prompt_id"], pw_df["sys1"], pw_df["sys2"])) if len(pw_df) else set()
new_rows = []

for _, prompt in prompts_df.iterrows():
    for s1, s2 in pairs:
        if (prompt["id"], s1, s2) in done_pairs: continue
        r1 = df[(df["prompt_id"] == prompt["id"]) & (df["system"] == s1)]
        r2 = df[(df["prompt_id"] == prompt["id"]) & (df["system"] == s2)]
        if len(r1) == 0 or len(r2) == 0: continue
        text1 = (r1["reasoning"].iloc[0] or "") + "\n\n" + (r1["recommendations"].iloc[0] or "")
        text2 = (r2["reasoning"].iloc[0] or "") + "\n\n" + (r2["recommendations"].iloc[0] or "")
        # Randomize order to avoid position bias
        flip = (prompt["id"] % 2 == 0)
        if flip: text1, text2 = text2, text1
        try:
            res = pairwise_judge(prompt["query"], text1, text2)
            winner = res.get("winner", "tie")
            if flip:
                winner = {"1": "2", "2": "1", "tie": "tie"}.get(winner, "tie")
            sys_winner = {"1": s1, "2": s2, "tie": "tie"}.get(winner, "tie")
            new_rows.append({"prompt_id": prompt["id"], "sys1": s1, "sys2": s2, "winner": sys_winner, "rationale": res.get("rationale", "")})
        except Exception as e:
            print(f"  ✗ pairwise prompt {prompt['id']} {s1} vs {s2}: {e}")

if new_rows:
    pw_df = pd.concat([pw_df, pd.DataFrame(new_rows)], ignore_index=True)
    pw_df.to_parquet(pairwise_path, index=False)

print(f"\n✅ Pairwise judgments: {len(pw_df)}")
# Win rates
print("\nWin rates:")
for s1, s2 in pairs:
    sub = pw_df[(pw_df["sys1"] == s1) & (pw_df["sys2"] == s2)]
    if len(sub) == 0: continue
    counts = sub["winner"].value_counts()
    print(f"  {s1} vs {s2}: {s1} wins {counts.get(s1,0)}  | {s2} wins {counts.get(s2,0)}  | ties {counts.get('tie',0)}")


### §8 — Observations + Decisions

- _Win rate B vs A: ... %_
- _If C present: B vs C ..._

---

# §9 — Human eval pilot (10 prompts)

**Why**: validates LLM judge against human judgment. You manually score 10 random
prompts, then we compute correlation. If r > 0.7 → LLM judge is reliable.

**How**: cell below generates a CSV `human_eval_template.csv` with 10 prompts × all systems
× 5 metrics. You fill the `score` column (1-5) by hand. Then we re-load and compare.

In [ ]:
# §9 — Generate human eval template
np.random.seed(SEED)
sample_ids = np.random.choice(prompts_df["id"].values, size=min(10, len(prompts_df)), replace=False)
sample_df = df[df["prompt_id"].isin(sample_ids)].sort_values(["prompt_id", "system"])

rows = []
for _, r in sample_df.iterrows():
    full_text = (r["reasoning"] or "") + "\n\n" + (r["recommendations"] or "")
    for metric in JUDGE_PROMPTS:
        rows.append({
            "prompt_id": r["prompt_id"],
            "system": r["system"],
            "metric": metric,
            "query": r["query"],
            "response": full_text[:500],
            "human_score": "",  # ← FILL THIS (1-5)
            "llm_score": judge_df[(judge_df["prompt_id"]==r["prompt_id"]) & (judge_df["system"]==r["system"]) & (judge_df["metric"]==metric)]["score"].iloc[0] if len(judge_df) else None,
        })

template = pd.DataFrame(rows)
template_path = OUTPUT_DIR / "human_eval_template.csv"
template.to_csv(template_path, index=False)
print(f"✅ Template saved: {template_path}")
print(f"   {len(template)} rows × {len(JUDGE_PROMPTS)} metrics × {template['system'].nunique()} systems")
print("\nNEXT: open the CSV in Excel/Numbers, fill the `human_score` column (1-5) for each row.")
print("      Then re-run the analysis cell below to get human-LLM correlation.")


In [ ]:
# §9b — Compute human vs LLM correlation (run AFTER filling the template)
human_path = OUTPUT_DIR / "human_eval_template.csv"
if human_path.exists():
    h = pd.read_csv(human_path)
    h["human_score"] = pd.to_numeric(h["human_score"], errors="coerce")
    h["llm_score"] = pd.to_numeric(h["llm_score"], errors="coerce")
    h_filled = h.dropna(subset=["human_score", "llm_score"])
    if len(h_filled) >= 5:
        corr = h_filled[["human_score", "llm_score"]].corr().iloc[0, 1]
        print(f"Human vs LLM correlation: r = {corr:.3f}  (n={len(h_filled)})")
        if corr > 0.7:
            print("✅ Strong agreement — LLM judge is reliable.")
        elif corr > 0.5:
            print("🟡 Moderate agreement — usable but interpret with caution.")
        else:
            print("⚠ Weak agreement — LLM judge results need expert validation.")
    else:
        print(f"Only {len(h_filled)} rows filled. Need ≥5 to compute meaningful correlation.")
else:
    print("Run §9 first to generate the template, then fill `human_score` column.")


### §9 — Observations + Decisions

- _Human vs LLM correlation r: ..._
- _Disagreements concentrated on which metric: ..._

---

# §10 — Composite score + visualizations (boxplot + heatmap + radar + win rate + scatter)

**Why**: synthesize all metrics into one number per system + 5 publication-ready figures.

In [ ]:
# §10a — Composite score (4-tier: L0/L1/L2/L3)
# Combines quantitative (MAPE inverted) + qualitative (judge mean) into single 0-100 score

quant_filt = pd.read_csv(OUTPUT_DIR / "quantitative_metrics_filtered.csv")
judge_df_full = pd.read_parquet(OUTPUT_DIR / "judge_scores.parquet")
qual_pivot = judge_df_full.groupby(["system", "metric"])["score"].mean().unstack()

# Composite = 50% accuracy (inverted MAPE, capped) + 50% qualitative (5 metrics avg)
def to_composite(row):
    mape = row.get("MAPE_median", np.nan)
    sys = row["system"]
    if sys not in qual_pivot.index:
        qual_score = 0
    else:
        qual_score = qual_pivot.loc[sys].mean()  # 0-5 scale
    qual_norm = (qual_score / 5) * 100  # 0-100
    if pd.isna(mape) or mape > 100:
        accuracy_norm = 0
    else:
        accuracy_norm = max(0, 100 - mape)  # higher is better
    return 0.5 * accuracy_norm + 0.5 * qual_norm

quant_filt["qual_mean"] = quant_filt["system"].map(lambda s: qual_pivot.loc[s].mean() if s in qual_pivot.index else np.nan)
quant_filt["composite_score"] = quant_filt.apply(to_composite, axis=1)
print("Composite scores (4-tier, FILTERED actual>=100):")
cols = ["system", "n_eval", "abstain_rate_pct", "MAPE_median", "qual_mean", "cost_total_usd", "composite_score"]
print(quant_filt[cols].to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
quant_filt.to_csv(OUTPUT_DIR / "composite_scores.csv", index=False)


In [ ]:
# §10b — 5 figures for Chapter 7 (4-tier visualizations)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
SYSTEMS = ["L0", "L1", "L2", "L3"]
COLORS = {"L0": "#888888", "L1": "#5B8DEF", "L2": "#F4A261", "L3": "#2A9D8F"}
PALETTE = [COLORS[s] for s in SYSTEMS]

# Reload data
df = pd.read_parquet(OUTPUT_DIR / "raw_responses.parquet")
df["pred_num"] = pd.to_numeric(df["predicted_sales_units"], errors="coerce")
df["actual_num"] = pd.to_numeric(df["actual_sales_units"], errors="coerce")
df_eval = df[(df["actual_num"] >= 100) & (~df["pred_num"].isna())].copy()
df_eval["ape"] = (df_eval["actual_num"] - df_eval["pred_num"]).abs() / df_eval["actual_num"] * 100
df_eval["sape"] = 2 * (df_eval["actual_num"] - df_eval["pred_num"]).abs() / (df_eval["actual_num"].abs() + df_eval["pred_num"].abs()) * 100

judge_df_full = pd.read_parquet(OUTPUT_DIR / "judge_scores.parquet")
qual_pivot = judge_df_full.groupby(["system", "metric"])["score"].mean().unstack()
quant_filt = pd.read_csv(OUTPUT_DIR / "quantitative_metrics_filtered.csv")

try:
    pairwise = pd.read_parquet(OUTPUT_DIR / "pairwise.parquet")
except FileNotFoundError:
    pairwise = pd.DataFrame()

# ── Figure 1: Boxplot of sMAPE per system (filtered) ───────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
data = [df_eval[df_eval["system"] == s]["sape"].values for s in SYSTEMS]
bp = ax.boxplot(data, labels=SYSTEMS, patch_artist=True, showmeans=True, meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 8})
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("sMAPE (%)")
ax.set_title("Forecast accuracy distribution by system (FILTERED actual≥100)\nLower is better. Diamond = mean.")
ax.set_ylim(0, max(200, df_eval["sape"].quantile(0.95) * 1.1))
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig1_smape_boxplot_4tier.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved fig1_smape_boxplot_4tier.png")

# ── Figure 2: Heatmap of judge scores (5 metrics × 4 systems) ──────────────
fig, ax = plt.subplots(figsize=(8, 4))
heatmap_data = qual_pivot.reindex(SYSTEMS)
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="RdYlGn", vmin=1, vmax=5, ax=ax,
            cbar_kws={"label": "Score (1-5)"}, linewidths=0.5)
ax.set_title("LLM-as-judge qualitative scores by system\nGreen = better. Higher = more grounded/actionable/etc.")
ax.set_xlabel("Quality dimension")
ax.set_ylabel("System")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig2_judge_heatmap_4tier.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved fig2_judge_heatmap_4tier.png")

# ── Figure 3: Radar chart (4 systems on 5 metrics) ─────────────────────────
import matplotlib.path as mpath
metrics = list(qual_pivot.columns)
N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": "polar"})
for sys in SYSTEMS:
    if sys not in qual_pivot.index: continue
    vals = qual_pivot.loc[sys, metrics].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=COLORS[sys], linewidth=2.5, label=sys)
    ax.fill(angles, vals, color=COLORS[sys], alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, size=10)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"], size=8)
ax.set_title("Qualitative dimensions — 4-tier comparison\n(Sonnet 4.6 LLM-as-judge, 1-5 scale)\n", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig3_radar_4tier.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved fig3_radar_4tier.png")

# ── Figure 4: Win-rate matrix (4×4 pairwise) ───────────────────────────────
if len(pairwise) > 0:
    fig, ax = plt.subplots(figsize=(6, 5))
    win_matrix = pd.DataFrame(0, index=SYSTEMS, columns=SYSTEMS, dtype=float)
    for sys_a in SYSTEMS:
        for sys_b in SYSTEMS:
            if sys_a == sys_b:
                win_matrix.loc[sys_a, sys_b] = np.nan
                continue
            sub = pairwise[((pairwise["system_a"] == sys_a) & (pairwise["system_b"] == sys_b)) |
                           ((pairwise["system_a"] == sys_b) & (pairwise["system_b"] == sys_a))]
            if len(sub) == 0:
                win_matrix.loc[sys_a, sys_b] = np.nan
                continue
            wins_a = ((sub["system_a"] == sys_a) & (sub["winner"] == "A")).sum() + ((sub["system_b"] == sys_a) & (sub["winner"] == "B")).sum()
            n_total = len(sub)
            win_matrix.loc[sys_a, sys_b] = wins_a / n_total * 100
    sns.heatmap(win_matrix, annot=True, fmt=".0f", cmap="RdYlGn", vmin=0, vmax=100, center=50,
                cbar_kws={"label": "Row's win rate vs Column (%)"}, ax=ax, linewidths=0.5)
    ax.set_title("Pairwise win rates (Sonnet 4.6 judge)\nRow > Column when value > 50.")
    ax.set_xlabel("Opponent system")
    ax.set_ylabel("Row system")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig4_winrate_matrix_4tier.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved fig4_winrate_matrix_4tier.png")

# ── Figure 5: Cost vs quality scatter ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5.5))
for _, r in quant_filt.iterrows():
    sys = r["system"]
    qual = qual_pivot.loc[sys].mean() if sys in qual_pivot.index else 0
    cost = r["cost_total_usd"]
    mape = r["MAPE_median"]
    size = 250 + (50 - mape if pd.notna(mape) else 50) * 5
    ax.scatter(cost, qual, s=size, color=COLORS[sys], alpha=0.7, edgecolors="black", linewidths=2, label=sys)
    ax.annotate(f"{sys}\nMAPE={mape:.1f}%" if pd.notna(mape) else f"{sys}\nMAPE=N/A",
                (cost, qual), xytext=(8, 8), textcoords="offset points", fontsize=10, fontweight="bold")
ax.set_xlabel("Total cost across 50 prompts (USD)")
ax.set_ylabel("Mean qualitative score (0-5)")
ax.set_title("Cost vs quality tradeoff (4-tier)\nBubble size ∝ accuracy. Top-left = cheap+good.")
ax.set_xlim(0, max(quant_filt["cost_total_usd"]) * 1.3)
ax.set_ylim(1, 4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig5_cost_quality_4tier.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved fig5_cost_quality_4tier.png")

print(f"\n✅ All 5 figures saved to {FIGURE_DIR}")


### §10 — Observations + Decisions

- _Composite winner: system X with score Y_
- _Most informative figure for Chapter 7: ..._

---

# §11 — Paste-ready Markdown tables for Chapter 7

**Why**: produces `chapter7_tables.md` ready to drop into the thesis.

In [ ]:
# §11 — Markdown summary tables for Chapter 7 (4-tier)
quant_filt = pd.read_csv(OUTPUT_DIR / "quantitative_metrics_filtered.csv")
quant_raw = pd.read_csv(OUTPUT_DIR / "quantitative_metrics_raw.csv")
judge_df_full = pd.read_parquet(OUTPUT_DIR / "judge_scores.parquet")
qual_pivot = judge_df_full.groupby(["system", "metric"])["score"].mean().unstack()
try:
    pairwise = pd.read_parquet(OUTPUT_DIR / "pairwise.parquet")
except FileNotFoundError:
    pairwise = pd.DataFrame()

print("# Chapter 7 — 4-tier agentic evaluation: ready-to-paste markdown\n")

# Table 1 — Quantitative metrics (FILTERED)
print("## Table 1 — Quantitative metrics (50 prompts, FILTERED actual≥100)\n")
print("| System | n_eval | abstain | MAPE | sMAPE | hallucination | latency p50 | cost (50 prompts) |")
print("|--------|-------:|--------:|-----:|------:|--------------:|------------:|------------------:|")
for _, r in quant_filt.iterrows():
    mape = f"{r['MAPE_median']:.2f}%" if pd.notna(r['MAPE_median']) else "n/a"
    smape = f"{r['sMAPE_median']:.2f}%" if pd.notna(r['sMAPE_median']) else "n/a"
    print(f"| **{r['system']}** | {int(r['n_eval'])} | {r['abstain_rate_pct']:.1f}% | {mape} | {smape} | {r['hallucination_rate_pct']:.1f}% | {r['latency_p50_s']:.2f}s | ${r['cost_total_usd']:.2f} |")

# Table 2 — Qualitative (LLM-as-judge)
print("\n## Table 2 — Qualitative scores (Sonnet 4.6 judge, 1-5 scale)\n")
metrics = list(qual_pivot.columns)
hdr = "| System | " + " | ".join(metrics) + " | **Mean** |"
print(hdr)
print("|" + "|".join(["--------"] + ["----:" for _ in metrics] + ["----:"]) + "|")
for sys in ["L0", "L1", "L2", "L3"]:
    if sys not in qual_pivot.index: continue
    row_vals = [f"{qual_pivot.loc[sys, m]:.2f}" for m in metrics]
    mean_score = qual_pivot.loc[sys].mean()
    print(f"| **{sys}** | " + " | ".join(row_vals) + f" | **{mean_score:.2f}** |")

# Table 3 — Pairwise win rates
if len(pairwise) > 0:
    print("\n## Table 3 — Pairwise win rates (78% complete, 233/300 judgments)\n")
    print("| Pair | A wins | B wins | Ties | Total | Winner |")
    print("|------|-------:|-------:|-----:|------:|:-------|")
    for sys_a, sys_b in [("L0","L1"),("L0","L2"),("L0","L3"),("L1","L2"),("L1","L3"),("L2","L3")]:
        sub = pairwise[(pairwise["system_a"] == sys_a) & (pairwise["system_b"] == sys_b)]
        if len(sub) == 0: continue
        wins_a = (sub["winner"] == "A").sum()
        wins_b = (sub["winner"] == "B").sum()
        ties = (sub["winner"] == "tie").sum()
        n = len(sub)
        if wins_a > wins_b * 1.2: winner = sys_a
        elif wins_b > wins_a * 1.2: winner = sys_b
        else: winner = "TIE"
        print(f"| {sys_a} vs {sys_b} | {wins_a} | {wins_b} | {ties} | {n} | **{winner}** |")

# Table 4 — Headline story
print("\n## Table 4 — Headline story per tier\n")
print("| Tier | Setup | What it tests | Key finding |")
print("|------|-------|---------------|-------------|")
print("| **L0** | LLM only, no tools | Pure language baseline | Honest abstain (100%); 0 forecasts produced |")
print("| **L1** | LLM + raw historical sales tool | Does data access alone help? | Yes — 26% MAPE on extrapolation; loses to L2/L3 in pairwise |")
print("| **L2** | LLM + UNTUNED ML (n_est=200, no Optuna) | Does ML structure help vs raw data? | Halves MAPE (24.74→13.27%); but qualitatively TIES with L1 (judge sees no clear improvement) |")
print("| **L3** | LLM + TUNED ML (Optuna 50-trial) + confidence layer | Does tuning + confidence-awareness help? | Lowest MAPE (9.64%); 0% hallucination; wins all 3 pairwise comparisons |")

print("\n## Headline sentence for thesis\n")
print('> "For €0.36 per query and +24% latency, the L3 system (LLM-orchestrated tuned ML with confidence-aware abstention) achieves **9.64% MAPE on filtered forecasts** with **0% hallucination rate**, ' \
      'outperforming a raw-data baseline (L1, 24.74% MAPE) and an untuned-ML baseline (L2, 13.27% MAPE) in both quantitative accuracy and Sonnet-4.6-judged composite quality."')


### §11 — Observations + Decisions

- All tables generated.
- Final composite winner: ...
- Headline narrative: ...

---

## End of A/B(/C) test notebook

**Outputs in `outputs_ab_test/`**:
- `prompts.csv` (50 prompts)
- `raw_responses.parquet` (50 × N_systems responses with reasoning, cost, latency)
- `quantitative_metrics.csv`
- `judge_scores.parquet`
- `pairwise.parquet`
- `composite_scores.csv`
- `chapter7_tables.md` (paste-ready for thesis)
- `figures/fig1-6.png` (6 publication figures)

**When Prometheus arrives**:
1. Add `PROMETHEUS_API_KEY` and `PROMETHEUS_ENDPOINT` to `.env`
2. Set `PROMETHEUS_API_AVAILABLE = True` in §4
3. Re-run §5 (only adds C, A and B already cached)
4. Re-run §6-§11 (incrementally updates all metrics)